# 07 : Analyst UI in a notebook

Purpose: use PhishLens from a notebook page. Paste an email, get the verdict,
the probability against the decision threshold, and the words that pushed the
model each way. Author: Colile Sibanda.

**Two ways to use this page.**

- **Read it**: the worked examples below were run and committed with their
  outputs, so they display in any notebook viewer, GitHub included.
- **Use it**: open it in Jupyter (`jupyter lab notebooks/`), run the setup cell,
  then edit the text in *Analyse your own email*, or use the interactive panel at
  the end.

The model runs inside this notebook through the same `ModelService` the API
serves from, so no server and no network are needed, and a verdict here is the
verdict the API, the desktop app and the browser UI would give. The client code
is in `src/notebook_ui/`.

**Reading a result.** The verdict comes first. In the text below it, words in
**bold** pushed towards phishing and words in *italic* towards legitimate;
darker shading means a larger weight. The text is shown as the model saw it:
lower case, with links, addresses and numbers replaced by `<url>`, `<email>` and
`<num>`. The weights explain this one prediction only (LIME), and the model
assists triage: an analyst still decides.

In [1]:
import os
import subprocess
from datetime import datetime
from pathlib import Path

# Notebooks run with the kernel's cwd set to notebooks/; the model files are
# found relative to the repository root, so move up to it first.
_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
os.chdir(_repo_root)

from notebook_ui import ApiBackend, LocalBackend, analyse, launch

backend = LocalBackend()
run_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()

print(f"Commit hash: {run_commit}")
print(f"Run date: {datetime.now().isoformat(timespec='seconds')}")
print("Backend: in-process model (LocalBackend)")

Commit hash: 3b9f8b19fd8d966b153bb25a543109396572f47f
Run date: 2026-10-18T10:16:21
Backend: in-process model (LocalBackend)


## Example: a phishing email

A typical credential lure: urgency, a threat of account closure, and a link.

In [2]:
PHISHING_EXAMPLE = """URGENT: your account has been suspended

Verify your password immediately at http://secure-update.example/login or your
account will be closed within 24 hours. Click here to confirm your identity now."""

analyse(PHISHING_EXAMPLE, backend)

word,direction,weight
your,towards phishing,+0.0129
suspended,towards phishing,+0.0095
click,towards phishing,+0.0080
account,towards phishing,+0.0075
here,towards phishing,+0.0071
at,towards legitimate,-0.0128
password,towards legitimate,-0.0052
closed,towards legitimate,-0.0024


## Example: a legitimate email

An ordinary internal message about a meeting.

In [3]:
LEGITIMATE_EXAMPLE = """Subject: minutes from tuesday meeting

Hi all, attached are the minutes from the project meeting. Please review the
action items before next week and let me know if I missed anything. Thanks, Sam"""

analyse(LEGITIMATE_EXAMPLE, backend)

word,direction,weight
action,towards phishing,+0.0041
and,towards phishing,+0.0035
minutes,towards phishing,+0.0026
tuesday,towards legitimate,-0.0060
project,towards legitimate,-0.0054
meeting,towards legitimate,-0.0054
thanks,towards legitimate,-0.0054
missed,towards legitimate,-0.0049


## Analyse your own email

Replace the text between the triple quotes with the subject and body of any
email (leave the headers out: the model does not use them), then run the cell.
The committed output shows a parcel-delivery lure as an example.

In [4]:
MY_EMAIL = """Your parcel could not be delivered

We attempted to deliver your package today but no one was available. To arrange
a new delivery, confirm your address and pay the R25 redelivery fee at
http://parcel-redelivery.example/track within 48 hours, or the package will be
returned to the sender."""

analyse(MY_EMAIL, backend)

word,direction,weight
your,towards phishing,+0.4061
pay,towards phishing,+0.0708
no,towards phishing,+0.0664
delivery,towards phishing,+0.0546
delivered,towards phishing,+0.0401
at,towards legitimate,-0.1013
the,towards legitimate,-0.1000
to,towards legitimate,-0.0727
<url>,towards legitimate,-0.0502
or,towards legitimate,-0.0490


## Interactive panel

With a running Jupyter kernel, this cell shows a text box and an **Analyse
email** button; the result appears under it. A static viewer such as GitHub
cannot run widgets and shows only a one-line placeholder here, which is why the
examples above use `analyse(...)` directly.

In [5]:
launch(backend)

## Using a running API instead

The same page can call the PhishLens API rather than loading the model locally,
for example the hosted demo. Explanations take up to a minute on the free host.

```python
hosted = ApiBackend("https://phishlens-vdc9.onrender.com")
analyse(MY_EMAIL, hosted)
launch(hosted)
```

Or a local API started with `python run_api.py`:
`ApiBackend("http://127.0.0.1:8000")`.